# Misinformation Models  — Opinion Classification
Driver notebook: runs each model, collects predictions, compares metrics.  
Add new models in the **Run models** cell. Train/dev only — test set not touched.

### Import configuration and packages ###

In [10]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch
import cnn_baseline as cnn

#Running this will import FastText vector file, which is stored on HuggingFace and is >4gb. 
from config import DATA_DIR, FASTTEXT_PATH

from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")

Device: cuda


In [11]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


### Run models ###


In [ ]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

In [ ]:
# Run CNN model here

# TextCNN
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
cnn_model    = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model    = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)
results["TextCNN"] = cnn.predict(cnn_model, dev_loader, DEVICE)



Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Epoch   1 | loss=0.7519 | dev_macro_f1=0.6922
Epoch   2 | loss=0.6382 | dev_macro_f1=0.7033
Epoch   3 | loss=0.5935 | dev_macro_f1=0.6985
Epoch   4 | loss=0.5460 | dev_macro_f1=0.6799
Epoch   5 | loss=0.4844 | dev_macro_f1=0.7166
Epoch   6 | loss=0.4333 | dev_macro_f1=0.6921
Epoch   7 | loss=0.3769 | dev_macro_f1=0.6995
Epoch   8 | loss=0.3272 | dev_macro_f1=0.6865
Epoch   9 | loss=0.2759 | dev_macro_f1=0.7064
Epoch  10 | loss=0.2473 | dev_macro_f1=0.7097
  Early stopping (no improvement for 5 epochs). Best F1: 0.7166


In [ ]:
# Logistic regression model

import logreg_baseline as lr
results["LogReg-Opinion"]  = lr.run(train_rows, dev_rows, task="opinion_label")


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013


In [14]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (not-op),F1 (opinion),AUC-ROC
Model,,,,,
LogReg-Opinion,0.7000,0.6931,0.7391,0.6471,0.7514


In [15]:
#Model details
# 
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


LogReg-Opinion
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    85      32
  true=1 (opinion):   28      55

              precision    recall  f1-score   support

 not-opinion       0.75      0.73      0.74       117
     opinion       0.63      0.66      0.65        83

    accuracy                           0.70       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.70      0.70      0.70       200


False Positives (predicted opinion, actually not) — 5 shown:
  [8] "No food, no FEMA: Hurricane Michael's survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw"
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTennis Student-